In [1]:
import sys
sys.path.append("..")
import numpy as np
import pickle

from lm_conf.default_utils.custom_types import OrganisedOutputs
from lm_conf.post_processing.metrics import BetaDistribution
from lm_conf.post_processing.metrics import dAUROC, dECE_equal_mass, dECE_equal_width, dAUROC_scalar, dECE_scalar

/home/ivan/miniconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = """
/hdd/ivny/results/trivia_qa/dist_lnll/meta-llama/Llama-3.1-8B-Instruct/2025-12-24_10-25-36
""".strip()
with open(f"{path}/graded_outputs_0.pkl", "rb") as f:
    graded_outputs: OrganisedOutputs = pickle.load(f)

In [3]:
conf: list[BetaDistribution] = graded_outputs.extracted_confidences[0]
acc: list = graded_outputs.accuracy_scores[0]
print(len(conf), len(acc))

17944 17944


In [4]:
# Clean accuracy to floats and filter invalid pairs

def to_float_acc(x):
    try:
        if x is None or x == "" or isinstance(x, list):
            return None
        val = float(x)
        if np.isnan(val):
            return None
        return val
    except (ValueError, TypeError):
        return None

clean_conf = []
clean_acc = []
for c, a in zip(conf, acc):
    fa = to_float_acc(a)
    if fa is None:
        continue
    if c is None:
        continue
    try:
        valid = c.is_valid()
    except Exception:
        valid = True
    if not valid:
        continue
    clean_conf.append(c)
    clean_acc.append(fa)

print("After cleaning:", len(clean_conf), len(clean_acc))

After cleaning: 16840 16840


In [ ]:
# train test split using cleaned data
from sklearn.model_selection import train_test_split

conf_train, conf_test, acc_train, acc_test = train_test_split(
    clean_conf,
    clean_acc,
    test_size=0.90,
    random_state=42,
)

print("conf", len(conf_train), len(conf_test))
print("acc", len(acc_train), len(acc_test))

conf 3368 13472
acc 3368 13472


In [6]:
def build_outputs(conf_list, acc_list):
    return OrganisedOutputs(
        extracted_confidences=[conf_list],
        accuracy_scores=[acc_list],
    )

In [7]:
# print("Uncalibrated Test dAUROC", dAUROC({}, build_outputs(conf_test, acc_test)))
# print("Uncalibrated Train dAUROC", dAUROC({}, build_outputs(conf_train, acc_train)))
print("Uncalibrated Test dAUROC scalar", dAUROC_scalar({}, build_outputs(conf_test, acc_test)))
print("Uncalibrated Train dAUROC scalar", dAUROC_scalar({}, build_outputs(conf_train, acc_train)))

[2025-12-31 03:21:29] INFO metrics.py:303: Computing dAUROC_scalar
[2025-12-31 03:21:29] INFO metrics.py:303: Computing dAUROC_scalar


Uncalibrated Test dAUROC scalar [0.8328339309483619]
Uncalibrated Train dAUROC scalar [0.832290637088016]


In [8]:
print("Uncalibrated Train dECE", dECE_scalar({"results_path": "."}, build_outputs(conf_train, acc_train)))
print("Uncalibrated Test dECE", dECE_scalar({"results_path": "."}, build_outputs(conf_test, acc_test)))

Uncalibrated Train dECE [0.26767056140046463]
Uncalibrated Test dECE [0.26522164690777067]


In [9]:
def construct_calibration_data(conf_lst: list[BetaDistribution], acc_lst: list[float]):
    # Step 1: Bin confidence scores into equal-width bins
    num_bins = 10
    bin_edges = np.linspace(0.0, 1.0, num_bins + 1)

    features = []
    targets = []
    target_dists = []

    # Bin the training data
    for i in range(num_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        
        # Include right edge for last bin
        if i == num_bins - 1:
            mask = np.array([(c.mu >= lo and c.mu <= hi) for c in conf_lst])
        else:
            mask = np.array([(c.mu >= lo and c.mu < hi) for c in conf_lst])

        conf_in_bin: list[BetaDistribution] = [conf_lst[j] for j in range(len(conf_lst)) if mask[j]]
        acc_in_bin: list[float] = [acc_lst[j] for j in range(len(acc_lst)) if mask[j]]
        print(conf_in_bin)
        # bin accuracy distribution
        if len(acc_in_bin) > 0:
            alpha_param = np.sum(acc_in_bin) + 1
            beta_param = len(acc_in_bin) - np.sum(acc_in_bin) + 1
            mu = alpha_param / (alpha_param + beta_param)
            sigma = np.sqrt((alpha_param * beta_param) / ((alpha_param + beta_param)**2 * (alpha_param + beta_param + 1)))
            bin_acc_dist = BetaDistribution(mu=mu, sigma=sigma)
            # build training data for this bin
            for i, c in enumerate(conf_in_bin):
                features.append((c.mu, c.sigma, c.alpha_param, c.beta_param, c.alpha_param + c.beta_param))
                target_dists.append((bin_acc_dist.mu, bin_acc_dist.sigma, bin_acc_dist.alpha_param, bin_acc_dist.beta_param, bin_acc_dist.alpha_param + bin_acc_dist.beta_param)) 
                targets.append(acc_in_bin[i])
        else:
            continue
    return features, targets, target_dists

X_train, y_train, y_train_dists = construct_calibration_data(conf_train, acc_train)
X_test, y_test, y_test_dists = construct_calibration_data(conf_test, acc_test)

[BetaDistribution(alpha=0.15492172981654043, beta=2.1934425171877927, mu=0.06597005980403796, sigma=0.13565552602195086), BetaDistribution(alpha=0.8666820989065147, beta=11.77773260844142, mu=0.06854268220124633, sigma=0.06840448407154874), BetaDistribution(alpha=0.8278353579737208, beta=7.76609995841546, mu=0.09632785534177643, sigma=0.09525395136889943), BetaDistribution(alpha=1.1981279252237325, beta=13.437548814063975, mu=0.0818635138344852, sigma=0.06933307668698956), BetaDistribution(alpha=0.39942629117916767, beta=3.7186802739511897, mu=0.096992704016275, sigma=0.13081589023818843), BetaDistribution(alpha=0.5838127098955089, beta=7.884820127193987, mu=0.0689382478997819, sigma=0.08233333847343983), BetaDistribution(alpha=0.4149577530034283, beta=4.955058311961176, mu=0.077273093410413, sigma=0.10579873566567898), BetaDistribution(alpha=0.8633762949614718, beta=9.666278076339731, mu=0.08199474213651517, sigma=0.08079927097036085), BetaDistribution(alpha=1.543351817935401, beta=15

### Calibration

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

def mean_concentration_calibration_scaling(
    features, targets, target_dist, X, eps=1e-6
):
    """
    Mean calibration via Platt Scaling (Logistic) and 
    Concentration calibration via Isotonic Regression.
    """

    def unpack(arr):
        mu    = np.array([x[0] for x in arr], dtype=float)
        sigma = np.array([x[1] for x in arr], dtype=float)
        conc  = np.array([x[4] for x in arr], dtype=float)
        return mu, sigma, conc

    # --- unpack ---
    mu_f, sigma_f, conc_f = unpack(features)
    mu_x, sigma_x, conc_x = unpack(X)
    _, _, conc_t = unpack(target_dist) # Ground truth concentrations

    # --- numerical safety ---
    mu_f = np.clip(mu_f, eps, 1 - eps)
    mu_x = np.clip(mu_x, eps, 1 - eps)
    sigma_f = np.clip(sigma_f, eps, None)
    sigma_x = np.clip(sigma_x, eps, None)

    # =====================
    # Mean calibration (Platt Scaling)
    # =====================
    logits_f = np.log(mu_f / (1 - mu_f))
    logits_x = np.log(mu_x / (1 - mu_x))
    
    Z_f = np.column_stack([logits_f, np.log(sigma_f)])
    Z_x = np.column_stack([logits_x, np.log(sigma_x)])

    lr = LogisticRegression(solver="lbfgs")
    lr.fit(Z_f, targets)
    mu_cal = lr.predict_proba(Z_x)[:, 1]

    # =========================
    # Concentration calibration (Isotonic)
    # =========================
    # IsotonicRegression fits a non-decreasing step function.
    # 'out_of_bounds=clip' ensures that if X has higher concentration 
    # than anything in 'features', it gets the max seen value.
    iso_reg = IsotonicRegression(out_of_bounds='clip')
    iso_reg.fit(conc_f, conc_t)
    
    conc_cal = iso_reg.predict(conc_x)
    
    # Ensure conc_cal doesn't hit zero (Beta dist requirement)
    conc_cal = np.clip(conc_cal, eps, None)

    # --- reconstruct Beta ---
    alpha_cal = mu_cal * conc_cal
    beta_cal  = (1 - mu_cal) * conc_cal
    
    # sigma = sqrt((alpha * beta) / (nu^2 * (nu + 1)))
    sigma_cal = np.sqrt(
        (alpha_cal * beta_cal) / ((conc_cal ** 2) * (conc_cal + 1))
    )

    return list(zip(mu_cal, sigma_cal, alpha_cal, beta_cal, conc_cal))

In [11]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import betaln, psi, expit

def kl_beta(alpha_p, beta_p, alpha_q, beta_q):
    """Analytic KL Divergence: KL(Target || Scaled)"""
    term1 = betaln(alpha_q, beta_q) - betaln(alpha_p, beta_p)
    term2 = (alpha_p - alpha_q) * psi(alpha_p)
    term3 = (beta_p - beta_q) * psi(beta_p)
    term4 = (alpha_q + beta_q - (alpha_p + beta_p)) * psi(alpha_p + beta_p)
    return term1 + term2 + term3 + term4

def parameterized_dual_stage_scaling_logistic_kl(
    features, targets, target_dist, X, eps=1e-6
):
    """
    PDSS with Bivariate Logistic Mean Scaling and KL Concentration Scaling.
    
    Stage 1: Location Scaling - Maps (mu, sigma) to targets via bivariate logistic.
    Stage 2: Dispersion Scaling - Optimizes concentration scaling via KL divergence.
    """
    def unpack(arr):
        arr = np.array(arr)
        # 0: mu, 1: sigma, 2: alpha, 3: beta, 4: conc
        return arr[:, 0], arr[:, 1], arr[:, 4]

    mu_f, sigma_f, conc_f = unpack(features)
    mu_x, sigma_x, conc_x = unpack(X)
    mu_t, _, conc_t = unpack(target_dist)
    y_f = np.array(targets)

    # ==========================================
    # STAGE 1: Location Scaling (Bivariate Platt)
    # ==========================================
    logits_f = np.log(np.clip(mu_f, eps, 1 - eps) / (1 - np.clip(mu_f, eps, 1 - eps)))
    logits_x = np.log(np.clip(mu_x, eps, 1 - eps) / (1 - np.clip(mu_x, eps, 1 - eps)))
    
    # Feature matrices: combining location (logit) and dispersion (log-sigma)
    Z_f = np.column_stack([logits_f, np.log(np.clip(sigma_f, eps, None))])
    Z_x = np.column_stack([logits_x, np.log(np.clip(sigma_x, eps, None))])
    
    def logistic_obj(params):
        # params: [w_logit, w_sigma, bias]
        w = params[:2]
        b = params[2]
        # dot product + bias
        z = np.dot(Z_f, w) + b
        p = expit(z)
        return -np.mean(y_f * np.log(p + eps) + (1 - y_f) * np.log(1 - p + eps))

    # Initialize with 1.0 for mu weight, 0.0 for sigma weight and bias
    res_mu = minimize(logistic_obj, x0=[1.0, 0.0, 0.0], method="L-BFGS-B")
    w_opt, b_opt = res_mu.x[:2], res_mu.x[2]
    
    mu_cal = expit(np.dot(Z_x, w_opt) + b_opt)

    # ==========================================
    # STAGE 2: Dispersion Scaling (KL Optimization)
    # ==========================================
    def concentration_obj(params):
        # params: [intercept, coefficient]
        a, b = params
        # Power-law relationship for concentration mapping
        conc_scaled = np.exp(a + b * np.log(np.clip(conc_f, eps, None)))
        
        alpha_s = mu_t * conc_scaled
        beta_s  = (1 - mu_t) * conc_scaled
        
        # Target distribution reconstructed from target_dist parameters
        alpha_t = mu_t * conc_t
        beta_t  = (1 - mu_t) * conc_t
        
        return np.mean(kl_beta(alpha_t, beta_t, alpha_s, beta_s))

    res_conc = minimize(concentration_obj, x0=[0.0, 1.0], method="L-BFGS-B")
    a_opt, b_opt = res_conc.x
    
    conc_cal = np.exp(a_opt + b_opt * np.log(np.clip(conc_x, eps, None)))

    # --- Reconstruction ---
    alpha_cal = mu_cal * conc_cal
    beta_cal  = (1 - mu_cal) * conc_cal
    sigma_cal = np.sqrt((alpha_cal * beta_cal) / ((conc_cal ** 2) * (conc_cal + 1)))

    return list(zip(mu_cal, sigma_cal, alpha_cal, beta_cal, conc_cal))

### Test

In [12]:
# calibrated_X = mean_concentration_calibration_scaling(X_train, y_train, y_train_dists, X_test)
calibrated_X = parameterized_dual_stage_scaling_logistic_kl(X_train, y_train, y_train_dists, X_test)
calibrated_distributions = [BetaDistribution(x[0], x[1]) for x in calibrated_X]

In [13]:
print("Uncalibrated Test dECE", dECE_equal_width({"results_path": "."}, build_outputs(conf_test, acc_test)))
print("Calibrated Test dECE", dECE_equal_width({"results_path": "."}, build_outputs(calibrated_distributions, y_test)))

[2025-12-31 03:21:30] INFO metrics.py:291: Computing dECE_equal_width
[2025-12-31 03:21:40] INFO metrics.py:285: Saved dECE distribution plots to ./dECE_equal_width_distributions_0.png
[2025-12-31 03:21:40] INFO metrics.py:291: Computing dECE_equal_width


Uncalibrated Test dECE [0.2790769849207507]


[2025-12-31 03:21:49] INFO metrics.py:285: Saved dECE distribution plots to ./dECE_equal_width_distributions_1.png


Calibrated Test dECE [0.02501581526767105]


In [14]:
print("Uncalibrated Test dECE", dECE_scalar({"results_path": "."}, build_outputs(conf_test, acc_test)))
print("Calibrated Test dECE", dECE_scalar({"results_path": "."}, build_outputs(calibrated_distributions, y_test)))

Uncalibrated Test dECE [0.26522164690777067]
Calibrated Test dECE [0.010475126211731084]


In [15]:
print("Uncalibrated Test dAUROC scalar", dAUROC_scalar({"results_path": "."}, build_outputs(conf_test, acc_test)))
print("Calibrated Test dAUROC scalar", dAUROC_scalar({"results_path": "."}, build_outputs(calibrated_distributions, y_test)))

[2025-12-31 03:21:49] INFO metrics.py:303: Computing dAUROC_scalar
[2025-12-31 03:21:50] INFO metrics.py:303: Computing dAUROC_scalar


Uncalibrated Test dAUROC scalar [0.8328339309483619]
Calibrated Test dAUROC scalar [0.8327560355478978]


In [16]:
# print("Uncalibrated Test dAUROC", dAUROC({"results_path": "."}, build_outputs(conf_test, acc_test)))
# print("Calibrated Test dAUROC", dAUROC({"results_path": "."}, build_outputs(calibrated_distributions, y_test)))